# 01 — Data Quality Assessment

This notebook runs the full DQA pipeline and documents every issue found in the raw dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Data & Inspect Shape / Dtypes

In [ ]:
from src.dqa import load_data, check_shape_and_types
df = load_data('../data/Student Depression Dataset.csv')
info = check_shape_and_types(df)
print(f"Shape: {df.shape}")
print(f"Memory: {info['memory_mb']:.2f} MB")
df.dtypes

## 2. Missing Values

In [ ]:
from src.dqa import check_missing, drop_missing_financial_stress
missing_report = check_missing(df)
display(missing_report)
df = drop_missing_financial_stress(df)
print(f"Shape after dropping null Financial Stress: {df.shape}")

## 3. Duplicate Rows

In [ ]:
from src.dqa import check_and_drop_duplicates
df = check_and_drop_duplicates(df)
print(f"Shape after dropping duplicates: {df.shape}")

## 4. CGPA = 0 Investigation

In [ ]:
from src.dqa import investigate_zero_cgpa
print(f"Rows with CGPA=0 BEFORE drop: {(df['CGPA']==0).sum()}")
df = investigate_zero_cgpa(df)
print(f"Shape after CGPA fix: {df.shape}")

## 5. Age Outlier Analysis (IQR)

In [ ]:
from src.dqa import age_outlier_analysis
fig, ax = plt.subplots(figsize=(8,4))
ax.boxplot(df['Age'].dropna(), vert=False)
ax.set_title('Age Distribution (boxplot)')
ax.set_xlabel('Age')
plt.tight_layout()
plt.savefig('../reports/figures/age_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
df = age_outlier_analysis(df)
print(f"Shape after Age outlier removal: {df.shape}")

## 6. Work/Study Hours = 0

In [ ]:
from src.dqa import check_zero_work_study_hours
df = check_zero_work_study_hours(df)
print(df['Work/Study Hours'].value_counts().head(10))

## 7. Rare 'Others' Category Handling

In [ ]:
from src.dqa import handle_rare_others
print("Sleep Duration counts before:", df['Sleep Duration'].value_counts().to_dict())
print("Dietary Habits counts before:", df['Dietary Habits'].value_counts().to_dict())
df = handle_rare_others(df)
print(f"\nShape after 'Others' removal: {df.shape}")

## 8. Near-Zero Variance — Work Pressure & Job Satisfaction

In [ ]:
from src.dqa import check_near_zero_variance
nzv_cols = check_near_zero_variance(df)
print(f"Near-zero variance columns to drop: {nzv_cols}")
for col in ['Work Pressure', 'Job Satisfaction']:
    if col in df.columns:
        print(f"  {col}: {df[col].value_counts(normalize=True).to_dict()}")

## 9. Class Balance

In [ ]:
from src.dqa import class_balance_report
balance = class_balance_report(df)
display(balance)

## 10. Save Cleaned Data & Print Summary

In [ ]:
df.to_csv('../data/cleaned.csv', index=False)
print(f"Saved cleaned.csv | Shape: {df.shape}")
print("\n=== DQA SUMMARY ===")
print(f"  Raw rows      : 27901")
print(f"  Clean rows    : {len(df)}")
print(f"  Columns       : {df.shape[1]}")
print(f"  Issues found  : missing FinancialStress, duplicates, CGPA=0, Age>60, Others categories, NZV columns")